In [2]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import KFold
import polars as pl
import os
from typing import Optional

## Gestures Application

### Initial Inference Test

In [3]:
circle_left_df = pl.read_csv("./gesture_data/gesture_data_1777951120.csv")

circle_left_df

timestamp,Imu0_linear_accleration_x,Imu0_linear_accleration_y,Imu0_linear_accleration_z,Imu0_angular_velocity_x,Imu0_angular_velocity_y,Imu0_angular_velocity_z
f64,f64,f64,f64,f64,f64,f64
1.729,0.1406,-1.0624,9.9075,-0.0105,-0.0817,-0.0567
1.757,0.0814,-1.0313,9.9338,-0.0389,-0.0542,-0.0615
1.785,-0.1346,-1.1079,9.9571,-0.0733,0.0026,-0.0246
1.813,-0.0431,-1.4201,10.1157,-0.0585,0.0192,0.0322
1.841,-0.0604,-1.2072,9.9882,0.0029,-0.0289,0.002
…,…,…,…,…,…,…
4.389,1.5093,-1.2592,10.2174,-0.0962,0.0377,0.025
4.417,1.3807,-0.3009,9.7095,0.2045,-0.0472,-0.0073
4.445,1.3663,0.0407,9.4636,0.1159,-0.0402,0.0067


In [4]:
def get_column_combos(imus, dims, readings):
    columns = [f"Imu{im}_{read}_{dim}" for im in imus for read in readings for dim in dims]
    return columns

imus = [0]
dims = ["x", "y", "z"]
readings = ["linear_accleration", "angular_velocity"]

X_one = circle_left_df.select(
    get_column_combos(imus, dims, readings)
).to_torch(dtype=pl.Float32)

X_one.shape

torch.Size([100, 6])

In [5]:
circle_left_label_df = pl.read_csv("./gesture_label/gesture_label_1777951120.csv")

circle_left_label_df

label
i64
1


In [14]:
circle_left_label_df.with_columns(
            (pl.col('label').replace(before, after) for before, after in [])
        ).to_torch()[0]

tensor([1])

In [6]:
# y_one = circle_left_label_df.with_columns(
#     pl.col('label').replace(10, 7)
# ).to_torch()[0]
y_one = circle_left_label_df.to_torch()[0]

y_one

tensor([1])

In [163]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_dim_1, num_layers, output_size, hidden_dim_2: int = 32, dropout_rate: float = 0.1):
        super(LSTMClassifier, self).__init__()
        self.hidden_dim_1 = hidden_dim_1
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size,
                            self.hidden_dim_1,
                            num_layers=num_layers,
                            batch_first=True
                           )
        self.dropout_1 = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(self.hidden_dim_1, output_size)
    
    def forward(self, x: torch.Tensor):
        h = torch.zeros(self.num_layers, x.size(0), self.hidden_dim_1).to(x.device)
        c = torch.zeros(self.num_layers, x.size(0), self.hidden_dim_1).to(x.device)
        out, _ = self.lstm(x, (h, c))
        out = out[:, -1, :]
        out = self.dropout_1(out)
        out = self.fc1(out)
        return out

In [100]:
classes = {
    'TAP': 0,
    'CIRCLE_LEFT': 1,
    'CIRCLE_RIGHT': 2
}

input_size = 6
hidden_size = 32
num_layers = 1
output_size = len(classes)

model = LSTMClassifier(input_size, hidden_size, num_layers, output_size)
model

LSTMClassifier(
  (lstm): LSTM(6, 32, batch_first=True)
  (dropout_1): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=32, out_features=3, bias=True)
)

In [97]:
X_one.unsqueeze(0).shape

torch.Size([1, 100, 6])

In [162]:
with torch.no_grad():
    predictions = model(X_one.unsqueeze(0))
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

Predicted Labels: tensor([0])


### Data Loading

In [164]:
class IMUDataSet(Dataset):
    def __init__(
            self,
            data_dir: str,
            labels_dir: str,
            class_mappings: list[tuple[int, int]] = [],
            mean: Optional[torch.Tensor] = None,
            std: Optional[torch.Tensor] = None,
            normalize: bool = False):
        super().__init__()
        self.data_dir = data_dir
        self.labels_dir = labels_dir
        self.data_paths = os.listdir(self.data_dir)
        self.label_paths = os.listdir(self.labels_dir)
        self.class_mappings = class_mappings
        self.mean = mean
        self.std = std
        self.normalize = normalize
        self.cache = {}
        self.imus = [0]
        self.dims = ["x", "y", "z"]
        self.readings = ["linear_accleration", "angular_velocity"]
        self.label_col = "label"
        self.epsilon = 1e-7

    def __len__(self):
        return len(self.data_paths)

    def __getitem__(self, idx):
        data, label = self.cache.get(idx, (None, None))
        if data is not None and label is not None:
            return data, label
        
        data_df = pl.read_csv(os.path.join(self.data_dir, self.data_paths[idx]))
        X_item = data_df.select(
            self.get_column_combos()
        ).to_torch(dtype=pl.Float32)
        
        label_df = pl.read_csv(os.path.join(self.labels_dir, self.label_paths[idx]))
        y_item = label_df.with_columns(
            (pl.col(self.label_col).replace(before, after) for before, after in self.class_mappings)
        ).to_torch()[0]

        if self.normalize and self.mean is not None and self.std is not None:
            X_item = (X_item - self.mean) / self.std

        try:
            self.cache[idx] = X_item, y_item
        except OSError:
            del self.cache[list(self.cache.keys())[0]]

        return X_item, y_item
    
    def get_column_combos(self):
        columns = [f"Imu{im}_{read}_{dim}" for im in self.imus for read in self.readings for dim in self.dims]
        return columns

In [165]:
def get_norm_values(data_loader: DataLoader):
    X_list = [X[0] for X, _ in data_loader]
    X_tensor = torch.cat(X_list)
    return X_tensor.mean(dim=0), X_tensor.std(dim=0)

temp_dataset = IMUDataSet("./gesture_data", "./gesture_label")

temp_loader = torch.utils.data.DataLoader(temp_dataset,
        batch_size=1,
        shuffle=False
    )

mean, std = get_norm_values(temp_loader)
print("Mean:", mean)
print("STD:", std)

Mean: tensor([-1.2777e+00,  1.6284e+00,  8.8419e+00,  2.3330e-02, -1.2615e-02,
        -8.3565e-03])
STD: tensor([5.8085, 4.2330, 4.7141, 1.2179, 0.8883, 0.9707])


In [344]:
def pad_collate(batch):
    tensors, targets = zip(*batch)
    data = pad_sequence(tensors, batch_first=True, padding_side="left")
    labels = torch.cat(targets)
    return data, labels

batch_size = 32

full_dataset = IMUDataSet(
    "./gesture_data", 
    "./gesture_label",
    mean=mean,
    std=std,
    normalize=True
)

### Training

In [341]:
def reset_weights(m):
  '''
    Try resetting model weights to avoid
    weight leakage.
  '''
  for layer in m.children():
    if hasattr(layer, 'reset_parameters'):
        print(f'Reset trainable parameters of layer = {layer}')
        layer.reset_parameters()

print(classes)

input_size = 6
hidden_size = 4
num_layers = 1
output_size = len(classes)

{'TAP': 0, 'CIRCLE_LEFT': 1, 'CIRCLE_RIGHT': 2}


In [345]:
import copy # Needed to save the best weights in memory

k_folds = 5
num_epochs = 200
learning_rate = 0.001
dropout_rate = 0.0
weight_decay = 0
scheduler_patience = 15
early_stopping_patience = 30

results = {}

criterion = nn.CrossEntropyLoss()
kfold = KFold(n_splits=k_folds, shuffle=True)

print('--------------------------------')

for fold, (train_ids, test_ids) in enumerate(kfold.split(full_dataset)):

    print(f'FOLD {fold}')
    print('--------------------------------')

    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

    trainloader = torch.utils.data.DataLoader(
                        full_dataset,
                        batch_size=batch_size,
                        collate_fn=pad_collate,
                        sampler=train_subsampler
    )
    # This acts as our validation loader during training
    testloader = torch.utils.data.DataLoader(
                        full_dataset,
                        batch_size=batch_size,
                        collate_fn=pad_collate,
                        sampler=test_subsampler
    )

    model = LSTMClassifier(input_size, hidden_size, num_layers, output_size, dropout_rate=dropout_rate)
    model.apply(reset_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # 1. Initialize the Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=scheduler_patience)
    
    # 2. Initialize Early Stopping Trackers
    epochs_no_improve = 0
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(0, num_epochs):
        
        # --- TRAINING PHASE ---
        model.train() # Turn ON Dropout
        current_train_loss = 0.0
        
        for i, data in enumerate(trainloader, 0):
            inputs, targets = data
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            current_train_loss += loss.item()
            
        avg_train_loss = current_train_loss / len(trainloader)

        # --- VALIDATION PHASE ---
        model.eval() # Turn OFF Dropout for validation
        current_val_loss = 0.0
        
        with torch.no_grad():
            for i, data in enumerate(testloader, 0):
                inputs, targets = data
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                current_val_loss += loss.item()
                
        avg_val_loss = current_val_loss / len(testloader)
        
        # Print epoch summary
        if epoch % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

        # 3. Step the Scheduler using the Validation Loss
        scheduler.step(avg_val_loss)

        # 4. Check Early Stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            # Save the exact weights that got this best score
            best_model_wts = copy.deepcopy(model.state_dict()) 
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stopping_patience:
                print(f'\nEarly stopping triggered at epoch {epoch}! Reverting to best weights.')
                break
            
    print('Training process has finished for this fold.')

    # Load the best weights we found during early stopping before running the final test
    model.load_state_dict(best_model_wts)
    
    # Saving the BEST model to disk
    save_path = f'./model/model-fold-{fold}.pth'
    torch.save(model.state_dict(), save_path)

    # --- FINAL TESTING PHASE ---
    print('Starting final fold evaluation...')
    model.eval()
    correct, total = 0, 0
    
    with torch.no_grad():
        for i, data in enumerate(testloader, 0):
            inputs, targets = data
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    accuracy = 100.0 * correct / total
    print('Accuracy for fold %d: %d %%' % (fold, accuracy))
    print('--------------------------------')
    results[fold] = accuracy

--------------------------------
FOLD 0
--------------------------------
Reset trainable parameters of layer = LSTM(6, 4, batch_first=True)
Reset trainable parameters of layer = Linear(in_features=4, out_features=3, bias=True)
Epoch   0 | Train Loss: 1.1576 | Val Loss: 1.1118
Epoch   5 | Train Loss: 1.1504 | Val Loss: 1.1065
Epoch  10 | Train Loss: 1.1418 | Val Loss: 1.1020
Epoch  15 | Train Loss: 1.1367 | Val Loss: 1.0979
Epoch  20 | Train Loss: 1.1253 | Val Loss: 1.0939
Epoch  25 | Train Loss: 1.1189 | Val Loss: 1.0902
Epoch  30 | Train Loss: 1.1147 | Val Loss: 1.0864
Epoch  35 | Train Loss: 1.1055 | Val Loss: 1.0827
Epoch  40 | Train Loss: 1.0968 | Val Loss: 1.0800
Epoch  45 | Train Loss: 1.0916 | Val Loss: 1.0787
Epoch  50 | Train Loss: 1.0867 | Val Loss: 1.0792
Epoch  55 | Train Loss: 1.0831 | Val Loss: 1.0803
Epoch  60 | Train Loss: 1.0828 | Val Loss: 1.0818
Epoch  65 | Train Loss: 1.0851 | Val Loss: 1.0825
Epoch  70 | Train Loss: 1.0829 | Val Loss: 1.0832
Epoch  75 | Train Loss:

In [334]:
fold = 4
save_path = f'./model/model-fold-{fold}.pth'
state_dict = torch.load(save_path, weights_only=True)
model = LSTMClassifier(input_size, hidden_size, num_layers, output_size, dropout_rate=dropout_rate)
model.load_state_dict(state_dict)
model.eval()
model

LSTMClassifier(
  (lstm): LSTM(6, 1, batch_first=True)
  (dropout_1): Dropout(p=0.0, inplace=False)
  (fc1): Linear(in_features=1, out_features=3, bias=True)
)

In [327]:
print(X_one.shape)
y_one

torch.Size([100, 6])


tensor([1])

In [335]:
#to_pred = ((X_one - min_val) / (max_val - min_val + 1e-7)).unsqueeze(0)
to_pred = ((X_one - mean) / std).unsqueeze(0)
#to_pred = X_one.unsqueeze(0)
with torch.no_grad():
    predictions = model(to_pred)
    print(predictions)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

tensor([[-0.0541,  0.3781,  0.2996]])
Predicted Labels: tensor([1])


In [336]:
X_another = pl.read_csv("./gesture_data/gesture_data_1777952087.csv").select(
    get_column_combos(imus, dims, readings)
).to_torch(dtype=pl.Float32)

y_another = pl.read_csv("./gesture_label/gesture_label_1777952087.csv").to_torch()[0]

print(X_another.shape)
y_another

torch.Size([100, 6])


tensor([2])

In [337]:
#to_pred = ((X_another - min_val) / (max_val - min_val + 1e-7)).unsqueeze(0)
to_pred = ((X_another - mean) / std).unsqueeze(0)
#to_pred = X_another.unsqueeze(0)
with torch.no_grad():
    predictions = model(to_pred)
    print(predictions)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

tensor([[-0.0480,  0.3356,  0.3115]])
Predicted Labels: tensor([1])


In [338]:
X_another = pl.read_csv("./gesture_data/gesture_data_1777952614.csv").select(
    get_column_combos(imus, dims, readings)
).to_torch(dtype=pl.Float32)

y_another = pl.read_csv("./gesture_label/gesture_label_1777952614.csv").to_torch()[0]

print(X_another.shape)
y_another

torch.Size([100, 6])


tensor([0])

In [339]:
#to_pred = ((X_another - min_val) / (max_val - min_val + 1e-7)).unsqueeze(0)
to_pred = ((X_another - mean) / std).unsqueeze(0)
#to_pred = X_another.unsqueeze(0)
with torch.no_grad():
    predictions = model(to_pred)
    print(predictions)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

tensor([[-0.0486,  0.3398,  0.3103]])
Predicted Labels: tensor([1])


## ONNX Export

In [296]:
X_one.unsqueeze(0).shape

torch.Size([1, 100, 6])

In [ ]:
model_path = "./model/gesture_rec_static.onnx"
input_name = "readings"
output_name = "gesture"
input_tensor = torch.randn(1, 100, 6)

model.eval()

torch.onnx.export(
    model,
    input_tensor,
    model_path,
    input_names = [input_name],
    output_names = [output_name],
    dynamo=False,
    external_data=False
)

/home/kojo/tmp/ipykernel_36500/1916155734.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/home/kojo/Code/730SemesterProject/exploration/imenv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4463: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


> Need to run onnxsim after this:

```bash
onnxsim ./model/gesture_rec_static.onnx ./model/gesture_rec_static_sim.onnx
```